# پروژهٔ دستیار پیگیری پرونده — معماری دوم

راهنما: این نوت‌بوک مانند تمرین‌ها، داده، چانک، کش، بازیابی ترکیبی، بازرتبه‌بندی، تولید، داوری و اقدام با تأیید انسانی را جدا نشان می‌دهد. حالت پیش‌فرض رایگان است. امبدینگ و داور آفلاین، بدل آزمایشی‌اند و کیفیت مدل واقعی را ثابت نمی‌کنند.

In [1]:
from pathlib import Path
import sys, json, subprocess, tempfile, hashlib
ROOT = Path.cwd()
assert (ROOT / 'src/casepilot').exists(), 'نوت‌بوک را از ریشهٔ پروژه اجرا کنید.'
sys.path.insert(0, str(ROOT / 'src'))
from casepilot.common import read_json
from casepilot.store import Store
from casepilot.agent import Agent
from casepilot.model import ReplayClient
from casepilot.retrieval import Retriever
from casepilot.hybrid import HybridRetriever
print('وضعیت: معماری دوم؛ اجرای پیش‌فرض بدون شبکه است.')

وضعیت: معماری دوم؛ اجرای پیش‌فرض بدون شبکه است.


## مرحلهٔ داده و چانک‌سازی

توضیح: ایندکس جدید از همان منابع غربال‌شده ساخته می‌شود. خانواده‌های ارزیابی و پاراگراف‌های پاسخ مستقیم وارد آن نمی‌شوند. عنوان، بخش، خط، نسخه و هش حفظ می‌شوند. کد و جدول بلوک اتمی‌اند؛ بلوک بیش‌ازحد بزرگ هنگام بسته‌بندی شاهد کنار می‌رود. تخمین توکن با شمار بایت محافظه‌کارانه است.

In [2]:
index = read_json(ROOT / 'data/index_v2_manifest.json')
assert hashlib.sha256((ROOT / 'data/corpus_v2.json').read_bytes()).hexdigest() == index['corpus_sha256']
print(json.dumps(index, ensure_ascii=False, indent=2))
print('توضیح: ایندکس ثابت موجود خوانده شد؛ ساخت دوباره اختیاری با ابزار build_index.py است و صورت قفل ارزیابی را تغییر می‌دهد.')

{
  "architecture": "v2",
  "chunk_config": {
    "name": "section_blocks_v2",
    "target_tokens": 1600,
    "overlap_tokens": 256,
    "token_estimator": "utf8_bytes_upper_bound"
  },
  "sources": 231,
  "chunks": 2116,
  "corpus_sha256": "cae28a1a596a410e19d1130b1f60e9055a944f7657b51c911e51a7ad353725c6",
  "parent_corpus_sha256": "e4cc916355d43571bc3c7a42546da6b4c5515fd438e675f0bc128c7ce254e8a0",
  "excluded_issue_families": [
    2311,
    3122,
    6109,
    6182,
    6215,
    6237,
    6525,
    6620,
    7336,
    7338,
    8151,
    8534,
    8608,
    8815,
    9218,
    10049,
    10615,
    10701,
    11157,
    11528,
    11920,
    11922,
    12150,
    12189,
    12432,
    12435,
    12607,
    13002,
    14031,
    14593,
    14639,
    14710,
    14793,
    16269,
    16375,
    16388,
    16474,
    16495,
    16732,
    16914,
    16977,
    17093,
    17106,
    17112,
    17119,
    17249,
    17250,
    17290
  ],
  "isolation": "Same screened source allowlist as

## مرحلهٔ بازیابی ترکیبی و کش

توضیح: جست‌وجوی واژگانی با امبدینگ واقعی در حالت زنده ترکیب می‌شود. ادغام رتبه با `RRF`، تنوع با `MMR` و هشدار نسخه اجرا می‌شوند. کش به متن نرمال‌شده و مدل وابسته است. در حالت زنده، ساخت کامل ایندکس یک مرحلهٔ مستقل و محدود است؛ هر پرسش فقط امبدینگ پرسش را می‌سازد.

In [3]:
client = ReplayClient()
retriever = HybridRetriever(client)
query = 'session_state widget value resets after navigation'
for method, rows in [('baseline', Retriever().search(query, method='baseline')), ('hybrid_fixture', retriever.search(query, k=8, version='1.49.0'))]:
    print('روش:', method, [(r['source_id'], r['section']) for r in rows[:5]])
retriever.search(query, k=8)
print('وضعیت کش پرسش تکراری:', retriever.last_trace['embedding'])
assert retriever.last_trace['embedding']['batches'] == 0

روش: baseline [('issue:11266', 'Summary'), ('issue:11266', 'Current Behavior'), ('issue:11266', 'Expected Behavior'), ('issue:7206', 'Checklist'), ('docs:widget-behavior', "Changing a widget's identity will reset it")]
روش: hybrid_fixture [('docs:session_state', 'Session State and Widget State association'), ('docs:session-state', 'Session State and Widget State association'), ('docs:widget-behavior', "Retain statefulness when changing a widget's identity"), ('docs:session_state', 'Caveats and limitations'), ('issue:11202', 'Summary')]
وضعیت کش پرسش تکراری: {'texts': 1, 'unique_missing': 0, 'cache_hits': 1, 'batches': 0, 'usage': [], 'identity': {'provider': 'offline-test-fixture', 'model': 'hash-features-v2', 'dimensions': 128, 'normalization': 'whitespace-v1'}}


## مرحلهٔ تولید، داوری و گفت‌وگو

توضیح: نقش‌های استخراج، بازرتبه‌بندی، تولید و داوری فراخوانی‌های جدا دارند. نسخهٔ اصلاح کیفیت، برنامهٔ تشخیص را در همان استخراج تولید می‌کند؛ گزارش اولیه، اصلاحات تازه و آزمایش‌های قبلی به زمینه می‌روند. منابع اداری فیلتر می‌شوند. داور برای ادعا و شاهد، عبارت واقعی پاسخ و دلیل پشتیبانی، و برای سؤال دلیل تازه‌بودن می‌دهد؛ کنترل کد قرارداد نامعتبر را رد می‌کند. بازنویسی جست‌وجوی فارسی حداکثر یک بار است. اصلاح حداکثر یک بار و با داوری دوباره انجام می‌شود. شکست به ارجاع امن می‌رسد. داور مدل جای بازبین انسانی نیست.

In [4]:
temporary = tempfile.TemporaryDirectory(prefix='casepilot-v2-notebook-')
store = Store(Path(temporary.name) / 'tracker.sqlite3')
agent = Agent(store=store, client=client, hybrid=retriever)
first = agent.turn('notebook-case', 'مقدار session_state پس از جابه‌جایی صفحه تغییر می‌کند.', 'nb-v2-1')
second = agent.turn('notebook-case', 'پاسخ: نسخه مشخص شد و نمونهٔ کوچک هم رفتار را دارد.', 'nb-v2-2', facts={'streamlit_version':'1.49.0','reproducible':True})
print(second['response'])
print('مسیر اجرا:', [s['stage'] for s in second['pipeline']])
print('داوری آزمایشی:', second['judge'])
assert second['model_calls'] <= 8 and second['repair_count'] <= 1
assert not store.get('notebook-case')['comments']

پاسخ پیشنهادی: اقدام بعدی: شاهد مستند را با نمونهٔ حداقلی مقایسه و نتیجه را اعلام کنید؛ همسان‌بودن علت هنوز ثابت نشده است.

شاهد مستند: متن رسمی؛ سازگاری نسخه باید بررسی شود.

بخش منبع:

```text
Navigating between pages
```

نسخهٔ منبع:

```text
695e6ce2fb1b23afc6df66b40420ab0013ad72fd
```

متن شاهد:

```text
In particular, clicking markdown links to other pages resets ``st.session_state``.
```

نشانی منبع:

```text
https://github.com/streamlit/docs/blob/695e6ce2fb1b23afc6df66b40420ab0013ad72fd/content/develop/concepts/multipage-apps/overview.md
```

شاهد مستند: متن رسمی؛ سازگاری نسخه باید بررسی شود.

بخش منبع:

```text
Session State and Widget State association
```

نسخهٔ منبع:

```text
695e6ce2fb1b23afc6df66b40420ab0013ad72fd
```

متن شاهد:

```text
Every widget with a key is automatically added to Session State:
```

نشانی منبع:

```text
https://github.com/streamlit/docs/blob/695e6ce2fb1b23afc6df66b40420ab0013ad72fd/content/develop/api-reference/caching-and-state/session_state.md
``

## مرحلهٔ بازبینی انسانی و اقدام

راهنما: متن و شاهدها را بخوانید و تصمیم را خودتان عوض کنید. پیش‌فرض رد است. تأیید به همین پرونده، هش و نسخهٔ پیشنهاد وابسته است؛ ویرایش یا اطلاعات تازه تأیید قبلی را نامعتبر می‌کند.

In [5]:
REVIEW_DECISION = 'reject'
p = second['proposal']
review = store.review('notebook-case', p['id'], p['hash'], REVIEW_DECISION, 'بازبین نوت‌بوک')
if REVIEW_DECISION == 'approve':
    result = store.execute('notebook-case', p['id'], review['approval_id'])
    assert store.execute('notebook-case', p['id'], review['approval_id'])['replayed']
    print(result)
else:
    assert not store.get('notebook-case')['comments']
    print('وضعیت: پیشنهاد رد شد و نظر ثبت نشد.')

وضعیت: پیشنهاد رد شد و نظر ثبت نشد.


## مرحلهٔ آزمون و ارزیابی توسعه

توضیح: آزمون‌ها خطاهای قرارداد، کش، بودجه، داور، اصلاح، استناد و مجوز را پوشش می‌دهند. مقایسهٔ حذف مؤلفه‌ها در ابزار ارزیابی فراهم است. ارزیابی آفلاین معیار کیفیت مدل نیست. آزمون قبلی دیده شده است؛ ادعای کیفیت نهایی معماری دوم به آزمون تازه و بازبینی مستقل نیاز دارد.

In [6]:
subprocess.run([sys.executable, 'scripts/run_tests.py'], cwd=ROOT, check=True)
configuration = hashlib.sha256(b''.join(p.read_bytes() for p in sorted((ROOT / 'src/casepilot').glob('*.py')))).hexdigest()[:12]
label = 'notebook_' + configuration
folder = ROOT / 'artifacts/architecture_v2/offline' / (label + '_full')
if not (folder / 'manifest.json').exists():
    subprocess.run([sys.executable, 'scripts/evaluate_v2.py', '--limit', '15', '--label', label], cwd=ROOT, check=True)
print('نتیجهٔ توسعهٔ آفلاین:', read_json(folder / 'manifest.json')['complete'])

نتیجهٔ توسعهٔ آفلاین: True


## مرحلهٔ کنترل خصمانه

توضیح: آزمون‌های مرز مجوز و نقل‌قولِ نسخهٔ اول حفظ شده‌اند؛ آزمون‌های معماری دوم، خروجی جعلی داور و بازرتبه‌بند، شکست داوری و توقف پس از یک اصلاح را جدا بررسی می‌کنند. این‌ها ادعای مقاومت مدل واقعی در برابر همهٔ تزریق‌ها نیستند.

In [7]:
subprocess.run([sys.executable, 'scripts/adversarial.py'], cwd=ROOT, check=True)
print('نتیجهٔ کنترل‌های قطعی:', read_json(ROOT / 'artifacts/test_results.json'))

نتیجهٔ کنترل‌های قطعی: {'at': '2026-10-07T20:21:11.293238+00:00', 'tests': 94, 'failures': 0, 'errors': 0, 'skipped': 0, 'passed': True, 'elapsed_seconds': 15.013324700004887}


## مرحلهٔ اجرای واقعی محدود

راهنما: کلید فقط در محیط فرایند باشد. اجرای زیر پیش‌فرض خاموش است. ساخت ایندکس سقف هشت سنت و نمونهٔ توسعه سقف هفت سنت دارد. راهنمای اجرا و تعرفه در مستندات آمده است. ارزیابی واقعی نسخهٔ اول و آزمایش معماری دوم نتایج جدا هستند.

In [8]:
RUN_LIVE = False
if RUN_LIVE:
    raise RuntimeError('برای اجرای پولی تازه، دفتر مشترک و سقف را طبق راهنمای اجرای واقعی بررسی کنید؛ ارزیابی ثابت را تکرار نکنید.')
else:
    saved = ROOT / 'artifacts/quality_revision/final_evaluation/completion_manifest.json'
    print('وضعیت: هیچ درخواست واقعی تازه‌ای اجرا نشد.')
    if saved.exists():
        measured = read_json(saved)
        semantic = read_json(saved.parent / 'ai_review_manifest.json')
        print('ارزیابی اصلاح کیفیت:', measured['complete'], 'مقایسه‌ها:', measured['attempted_comparisons'])
        print('سناریوهای عملیاتی:', measured['scenarios'])
        print('کیفیت معنایی؛ قضاوت هوش مصنوعی:', semantic['summary'])
        print('هزینهٔ افزودهٔ ارزیابی:', measured['new_charged_or_reserved_usd'])
    print('توضیح: نتیجه‌های معماری‌های قبلی تاریخی‌اند؛ مرور آن‌ها رایگان است. فرم انسانی خالی است.')
temporary.cleanup()

وضعیت: هیچ درخواست واقعی تازه‌ای اجرا نشد.
ارزیابی اصلاح کیفیت: True مقایسه‌ها: 40
سناریوهای عملیاتی: {'n': 10, 'passed': 8}
کیفیت معنایی؛ قضاوت هوش مصنوعی: {'dev': {'previous': {'n': 5, 'delivered': 5, 'overall_counts': {'0': 5}, 'validation_errors': {'none': 5}, 'median_latency_seconds': 12.512264, 'citations': 14}, 'revised': {'n': 5, 'delivered': 5, 'overall_counts': {'1': 4, '0': 1}, 'validation_errors': {'provider_error': 1, 'judge_rejected': 3, 'none': 1}, 'median_latency_seconds': 18.130561, 'citations': 1}}, 'test_fresh': {'previous': {'n': 15, 'delivered': 15, 'overall_counts': {'0': 7, '1': 8}, 'validation_errors': {'none': 12, 'judge_rejected': 2, 'provider_error': 1}, 'median_latency_seconds': 18.022422, 'citations': 26}, 'revised': {'n': 15, 'delivered': 15, 'overall_counts': {'1': 15}, 'validation_errors': {'judge_rejected': 13, 'none': 1, 'provider_error': 1}, 'median_latency_seconds': 35.28865, 'citations': 1}}}
هزینهٔ افزودهٔ ارزیابی: 0.4857199160000001
توضیح: نتیجه‌ه